# 🏢 AdventureWorks Enterprise Analytics Dashboard
## Week 3 – Friday Hackathon: Analytics Layer & Executive KPI Report

---

### 📋 Overview
This notebook connects exclusively to the **`analytics` schema** — a reusable intermediate analytics layer built on top of the AdventureWorks operational database.

**Pipeline Architecture:**
```
Raw Operational Tables (person, sales, production, humanresources, purchasing)
    ↓
Dimensional Layer (calendar_dim, product_dim, customer_dim, sales_person_dim)
    ↓
Fact Layer (sales_fact, purchasing_fact)
    ↓
Analytics Layer (monthly_revenue, customer_clv_segments, product_profitability,
                 territory_sales, salesperson_performance, inventory_health, vendor_scorecard)
    ↓
Executive KPI Layer (executive_dashboard_kpis)
    ↓
This Notebook ← Only reads from analytics.* views
```

**Business Domains Covered:** Sales · Customers · Products · Employees · Territories · Inventory · Purchasing

## 1. 🔌 Database Connection & Configuration

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sqlalchemy import create_engine
from IPython.display import Image, display

# ---- Connection Config ----
DB_HOST = 'localhost'
DB_PORT = 5432
DB_NAME = 'adventureworks'
DB_USER = 'postgres'
DB_PASS = 'Shayaan.xx5'

DATABASE_URL = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL)

# Test connection
with engine.connect() as conn:
    result = conn.execute(pd.io.sql.text('SELECT COUNT(*) FROM analytics.sales_fact'))
    count = result.fetchone()[0]
    print(f'✅ Connected to {DB_NAME}!')
    print(f'📊 Total sales transaction lines in analytics layer: {count:,}')

# ---- Plot Styling ----
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

CHARTS_DIR = 'charts'
os.makedirs(CHARTS_DIR, exist_ok=True)
print(f'📁 Charts will be saved to: {os.path.abspath(CHARTS_DIR)}')

## 2. 📦 Load Analytical Datasets from `analytics` Schema

> ⚠️ **All queries below target only the `analytics.*` views — no direct access to raw operational tables.**

In [ ]:
# Load all analytical datasets from the analytics schema
df_revenue = pd.read_sql("""
    SELECT year, month, month_name, monthly_revenue, monthly_gross_profit, 
           order_count, mom_revenue_growth_pct
    FROM analytics.monthly_revenue ORDER BY year, month;
""", engine)

df_territory = pd.read_sql("""
    SELECT territory_name, territory_group, total_revenue, total_orders, revenue_contribution_pct
    FROM analytics.territory_sales ORDER BY total_revenue DESC;
""", engine)

df_customer_seg = pd.read_sql("""
    SELECT customer_segment, COUNT(*) AS customer_count, 
           ROUND(SUM(customer_lifetime_value),2) AS total_clv
    FROM analytics.customer_clv_segments GROUP BY customer_segment ORDER BY total_clv DESC;
""", engine)

df_top_prod = pd.read_sql("""
    SELECT product_name, total_revenue, total_profit, realized_gross_margin_pct
    FROM analytics.product_profitability ORDER BY total_profit DESC LIMIT 10;
""", engine)

df_bottom_prod = pd.read_sql("""
    SELECT product_name, total_revenue, total_profit, realized_gross_margin_pct
    FROM analytics.product_profitability WHERE total_profit > 0 ORDER BY total_profit ASC LIMIT 10;
""", engine)

df_cat = pd.read_sql("""
    SELECT category_name, SUM(total_revenue) AS total_revenue, SUM(total_profit) AS total_profit,
           ROUND((SUM(total_profit)/NULLIF(SUM(total_revenue),0))*100,2) AS avg_margin
    FROM analytics.product_profitability WHERE category_name IS NOT NULL
    GROUP BY category_name ORDER BY total_revenue DESC;
""", engine)

df_emp = pd.read_sql("""
    SELECT salesperson_name, actual_sales_ytd, salesquota, quota_attainment_pct, total_profit, sales_rank
    FROM analytics.salesperson_performance WHERE actual_sales_ytd > 0 ORDER BY actual_sales_ytd DESC;
""", engine)

df_inv = pd.read_sql("""
    SELECT stock_status, COUNT(*) AS product_count, SUM(inventory_valuation) AS total_valuation
    FROM analytics.inventory_health GROUP BY stock_status ORDER BY total_valuation DESC;
""", engine)

df_vendor = pd.read_sql("""
    SELECT vendor_name, total_purchase_spend, avg_lead_time_days, rejection_rate_pct
    FROM analytics.vendor_scorecard ORDER BY total_purchase_spend DESC LIMIT 8;
""", engine)

df_kpi = pd.read_sql("""
    SELECT * FROM analytics.executive_dashboard_kpis ORDER BY year, month;
""", engine)

print('✅ All analytical datasets loaded successfully!')
print(f'  Revenue months: {len(df_revenue)} | Territories: {len(df_territory)} | Customer segments: {len(df_customer_seg)}')
print(f'  Products analyzed: {len(df_top_prod)+len(df_bottom_prod)} | Employees: {len(df_emp)} | Vendors (top 8): {len(df_vendor)}')

---
## 3. 📊 Visualizations & Business Insights

### Chart 1: Monthly Revenue Trend with MoM Growth

In [ ]:
df_revenue['date_label'] = df_revenue['year'].astype(str) + '-' + df_revenue['month'].astype(str).str.zfill(2)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

ax1.plot(df_revenue['date_label'], df_revenue['monthly_revenue'], marker='o', color='#1f77b4', linewidth=2.5, label='Monthly Revenue')
ax1.plot(df_revenue['date_label'], df_revenue['monthly_gross_profit'], marker='s', color='#2ca02c', linewidth=2, linestyle='--', label='Gross Profit')
ax2.bar(df_revenue['date_label'], df_revenue['mom_revenue_growth_pct'], color='#ff7f0e', alpha=0.3, width=0.4, label='MoM Growth %')

ax1.set_xlabel('Period (Year-Month)'); ax1.set_ylabel('Amount ($)', color='#1f77b4')
ax2.set_ylabel('MoM Growth (%)', color='#ff7f0e')
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
fig.suptitle('AdventureWorks Revenue & Gross Profit Trends with MoM Growth', fontsize=14, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
fig.tight_layout()
plt.savefig(f'{CHARTS_DIR}/1_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()

**📈 Business Insight — Revenue Trend:**

- Revenue shows a clear **seasonal growth pattern**, peaking in mid-year months, suggesting strong correlation with product release cycles or seasonal demand.
- Gross profit tracks closely with revenue but at a consistently lower margin, indicating **stable cost structures**.
- Months with **negative MoM growth** should trigger reviews of marketing spend, promotional effectiveness, or supply chain disruptions.
- Management can use this view to set realistic quarterly sales targets and allocate marketing budgets during historically low-revenue months.

### Chart 2: Sales Revenue by Territory

In [ ]:
plt.figure(figsize=(11, 6))
colors_map = {'North America': '#4472C4', 'Europe': '#ED7D31', 'Pacific': '#A9D18E'}
row_colors = df_territory['territory_group'].map(colors_map).fillna('#9E9E9E')

bars = plt.barh(df_territory['territory_name'], df_territory['total_revenue'], color=row_colors)
for bar in bars:
    w = bar.get_width()
    plt.text(w * 1.01, bar.get_y() + bar.get_height()/2, f'${w/1e6:.2f}M', va='center', fontsize=9, fontweight='semibold')

legend_handles = [mpatches.Patch(color=c, label=g) for g, c in colors_map.items()]
plt.legend(handles=legend_handles, title='Region')
plt.xlabel('Total Revenue ($)'); plt.ylabel('Territory')
plt.title('Revenue Contribution by Sales Territory', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis(); plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/2_sales_by_territory.png', dpi=150, bbox_inches='tight')
plt.show()

**🌍 Business Insight — Territory Performance:**

- **North America dominates** total revenue, with the Southwest and Northwest territories leading the pack.
- **European territories** show strong individual performance but collectively represent significant growth opportunity given their market size.
- The **Pacific territory** remains the smallest contributor — this warrants investigation into whether it's a resource allocation issue, market penetration challenge, or simply a smaller addressable market.
- Low-performing territories should receive dedicated sales headcount or targeted marketing campaigns to unlock latent demand.

### Chart 3: Customer RFM Segmentation

In [ ]:
colors_seg = ['#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.pie(df_customer_seg['customer_count'], labels=df_customer_seg['customer_segment'],
        autopct='%1.1f%%', colors=colors_seg, startangle=90,
        wedgeprops=dict(width=0.45, edgecolor='w'))
ax1.set_title('Customer Volume by Segment (Count Share)', fontsize=12, fontweight='bold')

ax2.pie(df_customer_seg['total_clv'], labels=df_customer_seg['customer_segment'],
        autopct='%1.1f%%', colors=colors_seg, startangle=90,
        wedgeprops=dict(width=0.45, edgecolor='w'))
ax2.set_title('Revenue Contribution by Customer Segment (Value Share)', fontsize=12, fontweight='bold')

fig.suptitle('Customer RFM Segmentation: Volume vs. Revenue Contribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/3_customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()

**👥 Business Insight — Customer Segmentation:**

- The **Platinum (VIP)** segment is small in count but contributes disproportionately to total revenue — a classic Pareto distribution.
- **Gold (Loyal)** customers represent the best growth opportunity: they already have high engagement and can be nurtured to VIP status through loyalty rewards and personalized outreach.
- **Bronze (Occasional)** customers make up a large portion of the headcount but a small share of revenue. Re-engagement campaigns (email promotions, seasonal discounts) can convert some to Silver.
- Focus retention spending on **Platinum + Gold** segments — losing even a few of these customers can significantly impact overall revenue.

### Chart 4: Product Profitability (Top 10 vs Bottom 10)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 11))

# Top 10
ax1.barh(df_top_prod['product_name'], df_top_prod['total_profit'], color='#2E8B57')
ax1.set_title('Top 10 Most Profitable Products', fontsize=12, fontweight='bold')
ax1.set_xlabel('Net Gross Profit ($)')
for p in ax1.patches:
    ax1.text(p.get_width()*1.01, p.get_y()+p.get_height()/2, f'${p.get_width()/1e3:.0f}K', va='center', fontsize=9)
ax1.invert_yaxis()

# Bottom 10
ax2.barh(df_bottom_prod['product_name'], df_bottom_prod['total_profit'], color='#CD5C5C')
ax2.set_title('Bottom 10 Lowest Profitable Products (with active sales)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Net Gross Profit ($)')
for p in ax2.patches:
    ax2.text(p.get_width()*1.01, p.get_y()+p.get_height()/2, f'${p.get_width():,.0f}', va='center', fontsize=9)
ax2.invert_yaxis()

fig.suptitle('Product Profitability Analysis — Best & Worst Performers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/4_product_performance.png', dpi=150, bbox_inches='tight')
plt.show()

**🛒 Business Insight — Product Performance:**

- The **top 10 products generate the vast majority of gross profit**, likely driven by high-margin road bikes and accessories.
- **Bottom performers** should be evaluated for potential discontinuation, price adjustment, or bundling strategies to improve margins.
- Products with revenue but near-zero profit indicate **pricing problems or excessive COGS** — a cost engineering review is recommended.
- The analytics layer allows product managers to track these rankings continuously without re-querying raw tables.

### Chart 5: Category Revenue & Margin Analysis

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

x = np.arange(len(df_cat['category_name'])); width = 0.35

b1 = ax1.bar(x - width/2, df_cat['total_revenue'], width, label='Revenue', color='#4682B4')
b2 = ax1.bar(x + width/2, df_cat['total_profit'], width, label='Gross Profit', color='#8FBC8F')
ax2.plot(x, df_cat['avg_margin'], color='#D2691E', marker='o', linewidth=2.5, label='Gross Margin %')

ax1.set_ylabel('Amount ($)'); ax2.set_ylabel('Gross Margin (%)')
ax1.set_xticks(x); ax1.set_xticklabels(df_cat['category_name'])
ax2.set_ylim(0, 100)

for rect in b1:
    h = rect.get_height()
    ax1.text(rect.get_x()+rect.get_width()/2, h*1.01, f'${h/1e6:.1f}M', ha='center', va='bottom', fontsize=9)

for i, v in enumerate(df_cat['avg_margin']):
    ax2.text(i, v+2, f'{v:.1f}%', ha='center', va='bottom', color='#D2691E', fontweight='bold')

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1+handles2, labels1+labels2, loc='upper right')
fig.suptitle('Category Financial Performance: Revenue, Profit & Gross Margin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/5_category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

**🏭 Business Insight — Category Performance:**

- **Bikes** generate the highest absolute revenue but may not have the best margin percentage — indicating a high-volume, moderate-margin model.
- **Accessories** often have higher margin percentages — a targeted upsell strategy during bike purchases can significantly improve overall profitability.
- **Components** are critical for revenue but require monitoring to ensure COGS don't erode margins as raw material prices fluctuate.
- Category-level margin analysis should guide pricing strategies, promotional discounts, and inventory stocking decisions.

### Chart 6: Salesperson Performance vs. Quota Targets

In [ ]:
plt.figure(figsize=(13, 6))
x = np.arange(len(df_emp['salesperson_name'])); width = 0.38

plt.bar(x - width/2, df_emp['actual_sales_ytd'], width, label='Actual Sales YTD', color='#6A5ACD')
plt.bar(x + width/2, df_emp['salesquota'], width, label='Annual Quota', color='#FF6347', alpha=0.7)

plt.xlabel('Sales Representative')
plt.ylabel('Amount ($)')
plt.title('Salesperson Actual Sales vs. Annual Quota Targets', fontsize=14, fontweight='bold')
plt.xticks(x, df_emp['salesperson_name'], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/6_employee_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# Show quota attainment table
print('\n📊 Quota Attainment Summary:')
display(df_emp[['salesperson_name','actual_sales_ytd','salesquota','quota_attainment_pct','sales_rank']]
        .rename(columns={'salesperson_name':'Name','actual_sales_ytd':'Actual Sales','salesquota':'Quota',
                         'quota_attainment_pct':'Attainment %','sales_rank':'Rank'}))

**👔 Business Insight — Salesperson Performance:**

- Top performers are dramatically **exceeding their quotas**, suggesting quotas may be set too conservatively for high performers — consider upward revision to drive ambition.
- Salespeople with **low quota attainment** should receive coaching, territory reassignment, or revised account portfolios.
- The **quota attainment % metric** allows HR and sales leadership to objectively benchmark performance for bonuses, promotions, and performance reviews.
- Consider cross-training top performers with underperformers to share best practices and customer relationship strategies.

### Chart 7: Inventory Health & Stock Status

In [ ]:
STATUS_COLORS = {
    'Healthy': '#4CAF50',
    'Overstocked': '#2196F3',
    'Below Reorder Point': '#FF9800',
    'Critical Stock Level': '#F44336',
    'Out of Stock': '#9C27B0'
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

row_colors_inv = df_inv['stock_status'].map(STATUS_COLORS).fillna('#9E9E9E')

ax1.bar(df_inv['stock_status'], df_inv['product_count'], color=row_colors_inv, edgecolor='white', linewidth=1.5)
ax1.set_title('Product Count by Stock Status', fontsize=12, fontweight='bold')
ax1.set_xlabel('Stock Status'); ax1.set_ylabel('Number of Products')
plt.setp(ax1.get_xticklabels(), rotation=25, ha='right')
for p in ax1.patches:
    ax1.text(p.get_x()+p.get_width()/2, p.get_height()+1, f'{int(p.get_height())}', ha='center', fontsize=10, fontweight='bold')

ax2.bar(df_inv['stock_status'], df_inv['total_valuation'], color=row_colors_inv, edgecolor='white', linewidth=1.5)
ax2.set_title('Inventory Capital Valuation by Status', fontsize=12, fontweight='bold')
ax2.set_xlabel('Stock Status'); ax2.set_ylabel('Total Valuation ($)')
plt.setp(ax2.get_xticklabels(), rotation=25, ha='right')
for p in ax2.patches:
    h = p.get_height()
    if h > 0:
        ax2.text(p.get_x()+p.get_width()/2, h*1.02, f'${h/1e6:.2f}M', ha='center', fontsize=9, fontweight='semibold')

legend_handles = [mpatches.Patch(color=c, label=s) for s, c in STATUS_COLORS.items()]
fig.legend(handles=legend_handles, loc='lower center', ncol=5, title='Stock Status Legend', bbox_to_anchor=(0.5, -0.05))
fig.suptitle('Inventory Health: Stock Levels & Capital Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/7_inventory_status.png', dpi=150, bbox_inches='tight')
plt.show()

**📦 Business Insight — Inventory Health:**

- A significant portion of inventory capital is tied up in **overstocked products** — this represents a carrying cost burden and potential obsolescence risk.
- Products flagged as **Critical Stock Level or Out of Stock** are potential lost-sale scenarios — procurement should prioritize these immediately.
- The **Inventory Health view** acts as an always-available early warning system that can power automated reorder alerts.
- Consider implementing an **ABC analysis** on top of this layer — high-value products (A) should have near-zero out-of-stock tolerance.

### Chart 8: Executive KPI — Supplier Scorecard (Spend vs. Defect Rate)

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

bars = ax1.bar(df_vendor['vendor_name'], df_vendor['total_purchase_spend'], color='#2F4F4F', label='Total Spend')
ax2.plot(df_vendor['vendor_name'], df_vendor['rejection_rate_pct'], color='#DC143C',
         marker='o', linewidth=2.5, label='Rejection Rate %')

ax1.set_ylabel('Total Purchase Spend ($)')
ax2.set_ylabel('Defect / Rejection Rate (%)')
ax1.tick_params(axis='x', rotation=35)
ax2.set_ylim(0, max(df_vendor['rejection_rate_pct']) * 1.5 + 2)

for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x()+bar.get_width()/2, h*1.01, f'${h/1e3:.1f}K', ha='center', va='bottom', fontsize=9)

for i, val in enumerate(df_vendor['rejection_rate_pct']):
    ax2.text(i, val+0.3, f'{val:.2f}%', ha='center', va='bottom', color='#DC143C', fontweight='semibold')

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1+handles2, labels1+labels2, loc='upper right')
fig.suptitle('Top 8 Suppliers: Purchase Spend vs. Product Defect/Rejection Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/8_executive_kpi_summary.png', dpi=150, bbox_inches='tight')
plt.show()

**🏭 Business Insight — Supplier Scorecard:**

- Suppliers receiving **high purchase spend** with **elevated rejection rates** represent a critical supply chain risk — quality issues from key suppliers can cascade into production delays and customer dissatisfaction.
- Vendors with **low rejection rates and high spend** are **preferred strategic partners** and should receive longer-term contracts.
- The rejection rate metric can be used as a **contract performance KPI** — vendors exceeding 5% rejection rates should face penalty clauses or volume reductions.
- Regular vendor scorecards like this should be shared with the procurement team as part of monthly supplier reviews.

---
## 4. 📋 Executive Summary KPI Table

In [ ]:
# Show executive KPI summary for latest periods
df_kpi_display = df_kpi[['year','month','month_name','total_revenue','total_gross_profit',
                           'total_orders','mom_revenue_growth_pct','active_customers','avg_clv',
                           'total_inventory_value','low_stock_product_count']].tail(12)

df_kpi_display = df_kpi_display.rename(columns={
    'year': 'Year', 'month': 'Mo', 'month_name': 'Month',
    'total_revenue': 'Revenue ($)', 'total_gross_profit': 'Gross Profit ($)',
    'total_orders': 'Orders', 'mom_revenue_growth_pct': 'MoM Growth %',
    'active_customers': 'Active Custs', 'avg_clv': 'Avg CLV ($)',
    'total_inventory_value': 'Inventory Value ($)', 'low_stock_product_count': 'Low Stock SKUs'
})

# Format currency
for col in ['Revenue ($)', 'Gross Profit ($)', 'Avg CLV ($)', 'Inventory Value ($)']:
    df_kpi_display[col] = df_kpi_display[col].apply(lambda x: f'${x:,.0f}')

print('📊 Executive Dashboard KPIs (Last 12 Months):')
display(df_kpi_display)

---
## 5. 🎯 Executive Recommendations

### 🟢 Five Business Opportunities

| # | Opportunity | SQL Evidence |
|---|-------------|-------------|
| 1 | **Upsell Accessories to Bike Buyers** — Accessories have high margins but low total revenue. Bundling strategies at point-of-sale can increase revenue per customer by 15-25%. | `analytics.product_profitability` shows high margin % for Accessories category |
| 2 | **European Market Expansion** — European territories significantly underperform relative to their market potential. Dedicated regional sales managers and localized marketing can unlock growth. | `analytics.territory_sales` shows Europe lagging North America |
| 3 | **Convert Silver → Gold Customers** — 'Silver (Steady)' customers are one loyalty program away from becoming high-value 'Gold' customers, dramatically improving CLV. | `analytics.customer_clv_segments` identifies this segment |
| 4 | **Preferred Vendor Program** — Consolidate purchasing with top-performing, low-rejection vendors to negotiate volume discounts and improve supply reliability. | `analytics.vendor_scorecard` identifies top-quality suppliers |
| 5 | **Seasonal Campaign Optimization** — Revenue shows clear seasonal patterns. Targeted pre-season marketing 6-8 weeks before peak months can amplify already strong periods. | `analytics.monthly_revenue` reveals seasonal peaks |

### 🔴 Five Business Risks

| # | Risk | SQL Evidence |
|---|------|--------------|
| 1 | **Over-Dependence on Top Customers** — Platinum VIP customers generate disproportionate revenue. Loss of even a few accounts could have severe financial impact. | `analytics.customer_clv_segments` concentration analysis |
| 2 | **Stock-Out Risk for Key Products** — Products flagged as 'Critical Stock Level' or 'Out of Stock' represent immediate lost sales and customer satisfaction risks. | `analytics.inventory_health` critical stock flags |
| 3 | **High Rejection Rate Suppliers** — One or more key suppliers have elevated defect rates. This risks production delays, quality complaints, and warranty costs. | `analytics.vendor_scorecard` rejection_rate_pct |
| 4 | **Quota Inflation for Low Performers** — Salespeople with very low quota attainment may be suffering from unrealistic targets or poor territory assignment, leading to attrition. | `analytics.salesperson_performance` quota_attainment_pct |
| 5 | **Overstocked Inventory Capital Lock-in** — Significant capital is tied up in overstocked items. This reduces cash flow flexibility and increases obsolescence risk. | `analytics.inventory_health` Overstocked valuation |

### ✅ Five Actionable Recommendations

| # | Recommendation | Priority |
|---|----------------|----------|
| 1 | **Implement a Tiered Customer Loyalty Program** — Create formal Bronze→Silver→Gold→Platinum upgrade paths with incentives at each tier. Monitor using `analytics.customer_clv_segments`. | HIGH |
| 2 | **Automate Reorder Alerts from Inventory Health View** — Build an automated nightly job that reads `analytics.inventory_health` and triggers purchase orders for all 'Below Reorder Point' and 'Critical' products. | HIGH |
| 3 | **Implement Vendor Quality Scoring in Procurement SLA** — Set contractual rejection rate caps (e.g., ≤3%) for all suppliers. Use `analytics.vendor_scorecard` for monthly compliance reviews. | MEDIUM |
| 4 | **Redesign Salesperson Quota Model** — Use historical performance data from `analytics.salesperson_performance` to set territory-specific, growth-adjusted quotas rather than flat annual targets. | MEDIUM |
| 5 | **Execute European Territory Growth Campaign** — Assign dedicated European account managers and run a localized digital campaign. Track results monthly via `analytics.territory_sales`. | HIGH |

---

## 🏁 Conclusion

This analytics pipeline successfully implements an **enterprise-grade reusable analytics layer** that:

- ✅ **14 analytical views** across 5 business domains (Sales, Customer, Product, Employee, Purchasing/Inventory)
- ✅ **Advanced SQL concepts**: Window Functions (`LAG`, `NTILE`, `DENSE_RANK`), CTEs, CASE WHEN, Conditional Aggregation, Complex JOINs
- ✅ **Chained dependency pipeline**: Every aggregate view builds upon dimensional/fact views
- ✅ **Zero raw table access** from this notebook — 100% via `analytics.*` schema
- ✅ **8 premium executive visualizations** with business insights
- ✅ **Actionable executive recommendations** supported by data evidence

> 💡 **Future-proofing**: Any new dashboard or report can be built by simply reading from the `analytics.*` schema — no raw table joins required.

---
*Generated by AdventureWorks Enterprise Analytics Pipeline | Week 3 – Friday Hackathon*